# Error taxonomy

Goal: find which kinds of financial language the frozen FinBERT CNN gets wrong, group them into an error taxonomy, and use the categories as subsets for evaluation and for targeted training (against random training and the LLM).

Pipeline:
1. **Error pool**: training sentences the CNN misclassifies, using its out-of-fold predictions from notebook 02 (`results/hard_samples/band_train.csv`), so every prediction comes from a model that never saw the sentence.
2. **Manual coding**: each teammate codes a sample of the pool with the categories in the codebook below; a shared subset measures agreement.
3. **Tagging at scale**: rules per category, checked against the hand-coded sentences, applied to all of train and val.
4. **Targeted vs random training per category**, and per-category evaluation (incl. the LLM).

Only the training set is used to build the taxonomy; val and test stay untouched for evaluation.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

sys.path.append("..")

pd.set_option("display.max_colwidth", None)
os.makedirs("../results/taxonomy", exist_ok=True)

## 1. Error pool
CNN out-of-fold predictions, hard mask and all-agree flag all come from notebook 02 (`results/hard_samples/band_train.csv`). `is_hard` and `all_agree` are kept in the pool for the analysis, but left off the coding sheets so they don't steer the coders.

In [2]:
train = pd.read_csv("../data/train.csv")
cnn = pd.read_csv("../results/hard_samples/band_train.csv")
assert (cnn.sentence.values == train.sentence.values).all()

errors = cnn[["sentence", "label", "pred", "max_prob", "is_hard", "all_agree"]].copy()
errors.insert(0, "train_idx", np.arange(len(train)))
errors = errors.rename(columns={"pred": "cnn_pred", "max_prob": "cnn_max_prob"})
errors["cnn_max_prob"] = errors.cnn_max_prob.round(3)
errors["n_words"] = train.sentence.str.split().str.len()

pool = errors[errors.cnn_pred != errors.label].copy()
pool["error_type"] = pool.label + " -> " + pool.cnn_pred
pool.to_csv("../results/taxonomy/error_pool.csv", index=False)

print(f"CNN wrong (pool): {len(pool)} / {len(train)}, of which hard: {pool.is_hard.sum()}, all-agree: {pool.all_agree.sum()}")
pool.error_type.value_counts()

CNN wrong (pool): 615 / 2901, of which hard: 193, all-agree: 114


error_type
positive -> neutral     273
neutral -> positive     123
negative -> neutral     100
negative -> positive     52
neutral -> negative      45
positive -> negative     22
Name: count, dtype: int64

## 2. Codebook

A sentence can get **several** categories. Mark a category with `1`.

| Column | Meaning | Example sentence |
|---|---|---|
| `ambiguous_label` | the sentence could reasonably get another label | "Sanoma will continue to focus on investing in digital media"; "However, the total orders received will still be above last year's levels" |
| `domain_required` | the sentiment needs domain knowledge to see | "Technopolis is Europe's leading operator of technoparks" |
| `mixed_sentiment` | positive and negative information in one sentence | "Profit up but sales down" |
| `corporate_action` | stock acquisition, company-internal events, collaboration | "Ponsse has agreed with Volvo on the start of cooperation in Latin America" |
| `numerical_comparison` | the sentiment depends on comparing numerical figures | "EPS stood at EUR0.07 versus EUR0.11" |
| `other` | none of the above: describe it in `notes` (candidates for new categories) | |

Rules:
- Code what makes the sentence **hard**, not every phenomenon it contains.
- `ambiguous_label` = label noise / disagreement. These are excluded from targeted training later, so use it honestly.
- Do not change the gold label; use `notes` if you think it is wrong.

In [3]:
CATEGORIES = ["ambiguous_label", "domain_required", "mixed_sentiment", "corporate_action",
              "numerical_comparison", "other"]
CODERS = ["A", "B", "C", "D"]
N_UNIQUE, N_SHARED = 50, 20   # per coder: 50 own sentences + 20 shared by everyone (for agreement)

# Stratified sample over error types, so the coded sentences have the same mix of mistakes as the pool.
sample, _ = train_test_split(pool, train_size=N_SHARED + N_UNIQUE * len(CODERS),
                             stratify=pool.error_type, random_state=42)
sample = sample.sample(frac=1, random_state=42)
shared = sample.iloc[:N_SHARED]
unique = sample.iloc[N_SHARED:]

columns = ["train_idx", "sentence", "label", "cnn_pred"]
for i, coder in enumerate(CODERS):
    path = f"../results/taxonomy/coding_sheet_{coder}.csv"
    if os.path.exists(path):   # never overwrite a sheet someone may already be coding
        print(f"{path} exists, not overwritten")
        continue
    sheet = pd.concat([shared, unique.iloc[i * N_UNIQUE:(i + 1) * N_UNIQUE]])[columns]
    sheet = sheet.sample(frac=1, random_state=i)          # shared rows mixed in, not all at the top
    sheet["shared"] = sheet.train_idx.isin(shared.train_idx)
    for category in CATEGORIES:
        sheet[category] = ""
    sheet["notes"] = ""
    sheet.to_csv(path, index=False)

print(f"{len(CODERS)} sheets x {N_UNIQUE + N_SHARED} sentences ({N_SHARED} shared), "
      f"{len(sample)} distinct sentences out of {len(pool)} in the pool")
sample.error_type.value_counts()

4 sheets x 70 sentences (20 shared), 220 distinct sentences out of 615 in the pool


error_type
positive -> neutral     98
neutral -> positive     44
negative -> neutral     36
negative -> positive    18
neutral -> negative     16
positive -> negative     8
Name: count, dtype: int64

## 3. Agreement (run after everyone has filled in their sheet)
Pairwise Cohen's kappa per category on the shared sentences, averaged over all coder pairs. Rough guide: > 0.6 good, 0.4-0.6 moderate, < 0.4 the category definition needs work.

In [ ]:
sheets = {c: pd.read_csv(f"../results/taxonomy/coding_sheet_{c}.csv") for c in CODERS}
shared_coded = {c: s[s.shared].set_index("train_idx").sort_index()[CATEGORIES].fillna(0).astype(int)
                for c, s in sheets.items()}

if all(s.to_numpy().sum() > 0 for s in shared_coded.values()):
    pairs = [(a, b) for i, a in enumerate(CODERS) for b in CODERS[i + 1:]]
    kappa = pd.Series({cat: np.mean([cohen_kappa_score(shared_coded[a][cat], shared_coded[b][cat]) for a, b in pairs])
                       for cat in CATEGORIES})
    display(kappa.round(2).sort_values())
else:
    print("Not all sheets are coded yet.")